# 05 — NER Fine-Tuning End-to-End

**Goal:** Fine-tune your first BERT NER model on a small CTI dataset. By the end you'll have a saved model that extracts `THREAT_ACTOR`, `MALWARE`, and `VULNERABILITY` entities.

## What you'll learn

1. Build a minimal CTI dataset in memory (so you can run this without downloads)
2. Load it into a `datasets.Dataset`
3. Tokenize + align labels (reusing the function from notebook 4)
4. Configure `TrainingArguments` and `Trainer`
5. Train and save
6. Run inference on new CTI sentences

> **Note on scale:** We'll use a ~20-sentence toy dataset. A real NER model needs hundreds to thousands of labeled examples. The workflow is identical; only the dataset size changes.

## Step 0 — Install dependencies

`datasets` and `seqeval` aren't always installed by default.

In [ ]:
# !pip install transformers torch datasets seqeval accelerate

## Step 1 — A tiny CTI dataset

Each example is a list of words and a list of BIO labels, same length. In a real project this comes from a labeled file (CoNLL, JSON, etc.).

In [1]:
raw_data = [
    {"words": ["APT28", "deployed", "X-Agent", "malware", "."],
     "labels": ["B-THREAT_ACTOR", "O", "B-MALWARE", "O", "O"]},
    {"words": ["The", "Lazarus", "Group", "used", "Cobalt", "Strike", "."],
     "labels": ["O", "B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "B-MALWARE", "I-MALWARE", "O"]},
    {"words": ["Attackers", "exploited", "CVE-2021-44228", "in", "Log4j", "."],
     "labels": ["O", "O", "B-VULNERABILITY", "O", "O", "O"]},
    {"words": ["Emotet", "spreads", "via", "phishing", "emails", "."],
     "labels": ["B-MALWARE", "O", "O", "O", "O", "O"]},
    {"words": ["Sandworm", "targeted", "Ukrainian", "infrastructure", "with", "Industroyer", "."],
     "labels": ["B-THREAT_ACTOR", "O", "O", "O", "O", "B-MALWARE", "O"]},
    {"words": ["The", "attack", "used", "CVE-2017-0199", "and", "CVE-2017-11882", "."],
     "labels": ["O", "O", "O", "B-VULNERABILITY", "O", "B-VULNERABILITY", "O"]},
    {"words": ["FIN7", "deployed", "Carbanak", "against", "financial", "targets", "."],
     "labels": ["B-THREAT_ACTOR", "O", "B-MALWARE", "O", "O", "O", "O"]},
    {"words": ["TrickBot", "and", "Ryuk", "were", "observed", "."],
     "labels": ["B-MALWARE", "O", "B-MALWARE", "O", "O", "O"]},
    {"words": ["APT29", ",", "also", "known", "as", "Cozy", "Bear", ",", "is", "active", "."],
     "labels": ["B-THREAT_ACTOR", "O", "O", "O", "O", "B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "O", "O", "O"]},
    {"words": ["Mimikatz", "extracted", "credentials", "during", "lateral", "movement", "."],
     "labels": ["B-MALWARE", "O", "O", "O", "O", "O", "O"]},
    {"words": ["The", "group", "exploited", "ProxyLogon", "(", "CVE-2021-26855", ")", "."],
     "labels": ["O", "O", "O", "O", "O", "B-VULNERABILITY", "O", "O"]},
    {"words": ["Conti", "ransomware", "encrypted", "systems", "."],
     "labels": ["B-MALWARE", "O", "O", "O", "O"]},
    {"words": ["Equation", "Group", "developed", "sophisticated", "implants", "."],
     "labels": ["B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "O", "O", "O"]},
    {"words": ["Stuxnet", "exploited", "four", "zero-days", "."],
     "labels": ["B-MALWARE", "O", "O", "O", "O"]},
    {"words": ["Attacks", "leveraged", "CVE-2014-0160", ",", "known", "as", "Heartbleed", "."],
     "labels": ["O", "O", "B-VULNERABILITY", "O", "O", "O", "O", "O"]},
    {"words": ["Turla", "operates", "from", "Russia", "and", "uses", "ComRAT", "."],
     "labels": ["B-THREAT_ACTOR", "O", "O", "O", "O", "O", "B-MALWARE", "O"]},
    {"words": ["Kimsuky", "is", "a", "North", "Korean", "actor", "."],
     "labels": ["B-THREAT_ACTOR", "O", "O", "O", "O", "O", "O"]},
    {"words": ["PlugX", "is", "a", "remote", "access", "trojan", "."],
     "labels": ["B-MALWARE", "O", "O", "O", "O", "O", "O"]},
    {"words": ["The", "exploit", "chain", "used", "CVE-2020-1472", "."],
     "labels": ["O", "O", "O", "O", "B-VULNERABILITY", "O"]},
    {"words": ["Charming", "Kitten", "deployed", "custom", "malware", "."],
     "labels": ["B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "O", "O", "O"]},
]

print(f"Dataset size: {len(raw_data)} examples")

Dataset size: 20 examples


## Step 2 — Load into `datasets.Dataset` and split

`Dataset` is Hugging Face's in-memory data structure. `Trainer` works with it natively.

In [3]:
from datasets import Dataset, DatasetDict

# Hugging Face Dataset.from_list wants a list of dicts — which is what we have
full = Dataset.from_list(raw_data)
split = full.train_test_split(test_size=0.25, seed=42)
ds = DatasetDict({"train": split["train"], "validation": split["test"]})
print(ds)

DatasetDict({
    train: Dataset({
        features: ['words', 'labels'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['words', 'labels'],
        num_rows: 5
    })
})


## Step 3 — Label maps and tokenizer

In [27]:
from transformers import AutoTokenizer

entity_types = ["THREAT_ACTOR", "MALWARE", "VULNERABILITY"]
label_list = ["O"] + [f"{p}-{t}" for t in entity_types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", label_list)

# Try all these two models and se the results
# bert-base-uncased
# distilbert-base-uncased


model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Labels: ['O', 'B-THREAT_ACTOR', 'I-THREAT_ACTOR', 'B-MALWARE', 'I-MALWARE', 'B-VULNERABILITY', 'I-VULNERABILITY']


## Step 4 — Tokenize + align labels

Same function as notebook 4, adapted for `datasets.map`.

In [28]:
def tokenize_and_align(examples):
    tokenized = tokenizer(examples["words"], is_split_into_words=True, truncation=True)
    new_labels = []
    for i, word_labels in enumerate(examples["labels"]):
        wids = tokenized.word_ids(batch_index=i)
        aligned = []
        previous = None
        for wid in wids:
            if wid is None:
                aligned.append(-100)
            elif wid != previous:
                aligned.append(label2id[word_labels[wid]])
            else:
                aligned.append(-100)
            previous = wid
        new_labels.append(aligned)
    tokenized["labels"] = new_labels
    return tokenized


tokenized_ds = ds.map(tokenize_and_align, batched=True, remove_columns=["words", "labels"])
print(tokenized_ds)
print("\nFirst tokenized example keys:", list(tokenized_ds["train"][0].keys()))

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5
    })
})

First tokenized example keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


## Step 5 — Data collator for dynamic padding

`DataCollatorForTokenClassification` pads each batch to its longest example (instead of padding the whole dataset to the global max) and handles the label `-100` padding correctly.

In [29]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Step 6 — Metric function (token-level F1 via seqeval)

Covered more deeply in notebook 6. Minimum here:

In [30]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [id2label[l] for l in label_seq if l != -100]
        for label_seq in labels
    ]
    true_preds = [
        [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

## Step 7 — Model, TrainingArguments, Trainer

In [31]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="./ner-cti-bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_11835/234787644.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.141486,0.000000,0.000000,0.000000
2,No log,0.754649,0.000000,0.000000,0.000000
3,1.279800,0.621355,0.000000,0.000000,0.000000
4,1.279800,0.555471,0.000000,0.000000,0.000000
5,0.561200,0.507396,0.400000,0.285714,0.333333
6,0.561200,0.477401,0.400000,0.285714,0.333333
7,0.561200,0.455247,0.400000,0.285714,0.333333
8,0.388200,0.448176,0.400000,0.285714,0.333333
9,0.388200,0.448249,0.400000,0.285714,0.333333
10,0.359500,0.447992,0.400000,0.285714,0.333333


/home/saqib/venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/saqib/venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=20, training_loss=0.6471475660800934, metrics={'train_runtime': 162.2323, 'train_samples_per_second': 0.925, 'train_steps_per_second': 0.123, 'total_flos': 1565299569966.0, 'train_loss': 0.6471475660800934, 'epoch': 10.0})

## Step 8 — Save and reload

In [33]:
trainer.save_model("./ner-cti-bert-final")
tokenizer.save_pretrained("./ner-cti-bert-final")

('./ner-cti-bert-final/tokenizer_config.json',
 './ner-cti-bert-final/special_tokens_map.json',
 './ner-cti-bert-final/vocab.txt',
 './ner-cti-bert-final/added_tokens.json',
 './ner-cti-bert-final/tokenizer.json')

## Step 9 — Inference with `pipeline`

In [34]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="./ner-cti-bert-final",
    tokenizer="./ner-cti-bert-final",
    aggregation_strategy="simple",
)

tests = [
    "APT28 deployed X-Agent malware exploiting CVE-2017-0199.",
    "Fancy Bear targeted NATO with custom implants.",
    "Emotet was observed dropping TrickBot and Cobalt Strike.",
]
for t in tests:
    print(t)
    for e in ner(t):
        print(f"  {e['entity_group']:<14} '{e['word']}' ({e['score']:.2f})")
    print()

Device set to use cuda:0


APT28 deployed X-Agent malware exploiting CVE-2017-0199.
  THREAT_ACTOR   'apt' (0.27)
  VULNERABILITY  'cv' (0.41)

Fancy Bear targeted NATO with custom implants.
  THREAT_ACTOR   'fancy' (0.27)

Emotet was observed dropping TrickBot and Cobalt Strike.
  THREAT_ACTOR   'em' (0.25)



## Expectations

With 20 examples you won't get production-quality results. Expect:
- Model picks up obvious patterns (CVE- prefix → `VULNERABILITY`)
- Misses names it has never seen (Fancy Bear) unless they resemble training examples
- F1 on validation will swing wildly because validation set is only ~5 examples

The point is the **workflow**, not the score. Real projects use DNRTI (~300 docs) or larger.

## Your turn — exercises

1. Double the dataset (write 20 more examples). Retrain. Does F1 improve?
2. Try `distilbert-base-uncased` as the checkpoint — it's faster. Does NER quality drop noticeably?
3. Swap in `ehsanaghaei/SecureBERT` and compare on the same dataset.

## Next notebook

**`06_ner_evaluation_with_seqeval.ipynb`** — the F1 number we saw is a summary. Now we'll break it down by entity type, do error analysis, and understand why span-level (not token-level) metrics are the right thing to report.